# 두 번째 독립 테스트셋 (MASSIVE) 평가 (논문에 없는 추가)
MASSIVE 1.1 영어 test 질의에 우리 함수 기준 정답을 붙인 셋(`data/massive_test.jsonl`, 465개: positive 233 / negative 232)으로
모델마다 생성·확률·채점을 하고, 거절 경계 지표(AUROC, 거절@recall90, negative 유형별, bootstrap 95% 구간)를 계산한다.
설계는 `EXPERIMENT_DESIGN.md` 3.1절, 셋 설명은 `MASSIVE_TEST.md`.

- 런타임: **A100 GPU**. 🔑 Secrets에 `HF_TOKEN`
- test 파일은 저장소에 들어 있다 (`git pull` 로 받음). Drive 에 따로 올릴 필요 없음
- 결과: Drive `MyDrive/octopus_v2/runs/eval_massive/<모델 이름>/` 에 `preds.jsonl`, `probs.jsonl`, `scored.jsonl`, `summary.json`
- 거절@recall90 의 문턱값은 **dev(우리 test 800개)** 에서 정한다. dev 확률은 `runs/eval/<모델 이름>/probs.jsonl` (5단계 노트북) 을 그대로 쓰고, 없으면 여기서 만든다
- 생성은 중간에 끊겨도 같은 셀을 다시 실행하면 남은 것만 이어서 한다

> ⚠️ 설계 문서 7절 5단계: MASSIVE·BFCL 은 **실험 모델이 모두 확정된 뒤 한 번** 채점하는 최종 test 다.
> 실험 A·B 모델을 만드는 중이라면 이 결과를 보고 데이터·설정을 바꾸지 않는다 (본 뒤에 바꾸면 변경 기록에 적는다).

## 준비 (그냥 실행)

In [ ]:
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
EVAL = f"{DRIVE}/runs/eval_massive"      # MASSIVE 결과
DEV_EVAL = f"{DRIVE}/runs/eval"          # dev(우리 test) 결과 (5단계 노트북과 같은 곳)
TEST = "data/massive_test.jsonl"
DEV_TEST = "data/test.jsonl"

# 평가할 모델: 이름 -> 학습한 폴더 또는 HF 이름. 이름은 결과 폴더 이름이 된다.
# 실험 A·B 모델이 생기면 여기에 줄을 더한다 (예: "A05": f"{DRIVE}/runs/A05").
MODELS = {
    "ours": f"{DRIVE}/runs/full",          # 4단계에서 학습한 모델 (= 실험의 A50, B0)
    "public": "NexaAIDev/Octopus-v2",      # 공개 모델
}
os.makedirs(EVAL, exist_ok=True)
!wc -l {TEST}
!python -c "import json,collections; r=[json.loads(l) for l in open('{TEST}')]; print(collections.Counter(x['kind'] for x in r), collections.Counter(x['neg_type'] for x in r if x['neg_type']))"

---
## 8-1. 생성 + 확률 + 채점 (모델마다)
**예측**: (돌리기 전에 적기) 예: 우리 모델은 질문형 positive·음악(서비스 없음)을 거절하고, 공개 모델은 near-miss(알람 조회·조명 등)에 함수를 부를 것.
> (여기에 예측)

In [ ]:
for name, path in MODELS.items():
    out = f"{EVAL}/{name}"
    print(f"===== {name}: {path}")
    !python -m src.evaluate generate --model "{path}" --test-file {TEST} --out {out}/preds.jsonl
    !python -m src.probs generate --model "{path}" --test-file {TEST} --preds {out}/preds.jsonl --out {out}/probs.jsonl
    !python -m src.evaluate score --preds {out}/preds.jsonl --test-file {TEST}

## 8-2. dev(우리 test) 확률 확인
거절@recall90 문턱값용. 5단계 노트북에서 만든 `runs/eval/<이름>/probs.jsonl` 이 있으면 그대로 쓰고, 없는 모델만 여기서 만든다 (800개, 시간이 걸림).

In [ ]:
for name, path in MODELS.items():
    dev = f"{DEV_EVAL}/{name}"
    if os.path.exists(f"{dev}/probs.jsonl") and sum(1 for _ in open(f"{dev}/probs.jsonl")) == 800:
        print(f"{name}: dev probs 있음 -> {dev}/probs.jsonl")
        continue
    print(f"===== {name}: dev probs 만들기")
    !python -m src.evaluate generate --model "{path}" --test-file {DEV_TEST} --out {dev}/preds.jsonl
    !python -m src.probs generate --model "{path}" --test-file {DEV_TEST} --preds {dev}/preds.jsonl --out {dev}/probs.jsonl

---
## 8-3. 나란히 보기 (기본 생성, greedy)

In [ ]:
import json
from src.evaluate import METRIC_NAMES

rows = {n: json.load(open(f"{EVAL}/{n}/summary.json", encoding="utf-8")) for n in MODELS}
print(f"{'':<36}" + "".join(f"{n:>12}" for n in rows))
for key, label in METRIC_NAMES.items():
    cells = [rows[n]["전체"][key] for n in rows]
    print(f"  {label:<34}" + "".join(f"{c:>12}" if key == "n" else f"{c:>12.2%}" for c in cells))

## 8-4. 거절 경계 지표 (설계 문서 6절)
- `auroc`: `<nexa_20>` 확률로 positive / negative 를 가르는 능력 (문턱값과 무관)
- `refuse@r`: dev positive 의 90% 가 함수 쪽으로 가게 하는 문턱값을 MASSIVE 에 그대로 썼을 때 negative 거절 비율. `pos_recall` 은 그때 MASSIVE positive 가 함수 쪽으로 간 비율
- `refuse:<유형>`: 같은 문턱값으로 negative 유형별 거절 비율 (`coding_tech` 는 1개뿐이라 의미 없음)
- 아래 "차이" 는 첫 번째 모델(`ours`) 기준, 같은 표본으로 뽑은 짝지은 bootstrap. `*` 는 95% 구간이 0 을 포함하지 않음

In [ ]:
args = " ".join(f'--model {n} {EVAL}/{n}/probs.jsonl {DEV_EVAL}/{n}/probs.jsonl' for n in MODELS)
!python -m src.refusal_metrics --test {TEST} {args} | tee {EVAL}/refusal_metrics.txt

### 틀린 near-miss 와 확신 정도
공개 모델이 BFCL 에서 near-miss 를 확신하며 틀린 것(선택 확률 중앙값 0.97)이 MASSIVE 에서도 나오는지 본다.

In [ ]:
import statistics
from src.preprocess.prompt import load_jsonl
from pathlib import Path

for n in MODELS:
    sc = load_jsonl(Path(f"{EVAL}/{n}/scored.jsonl"))
    pr = load_jsonl(Path(f"{EVAL}/{n}/probs.jsonl"))
    wrong = [(s, p) for s, p in zip(sc, pr) if s.get("neg_type") == "near_miss" and s["outcome"] == "function:false_call"]
    pc = [p["func"]["p_chosen"] for _, p in wrong if p.get("func")]
    print(f"== {n}: near-miss {sum(1 for s in sc if s.get('neg_type') == 'near_miss')}개 중 false_call {len(wrong)}개, "
          f"고른 함수 확률 중앙값 {statistics.median(pc) if pc else float('nan'):.3f}")
    for s, p in wrong[:15]:
        f = p.get("func") or {}
        print(f"   {f.get('chosen')} p={f.get('p_chosen', 0):.2f} | {s['query'][:80]}")

---
## 끝나면
결과는 Drive `runs/eval_massive/` 에 있다 (`refusal_metrics.txt` 에 지표 표). 랩탑에서 Claude 에게 "MASSIVE 결과 분석해줘" 라고 하면 Drive 에서 받아 오답 분석을 한다.